In [1]:
from google.colab import drive
drive.mount('/content/drive')

dir_ = '/content/drive/MyDrive/Cohand/Minh học data/Kỳ 3/NLP/'
raw_data = dir_ + 'data/raw/'
processed_data = dir_ + 'data/processed/'
requirment_file = dir_ + 'requirements.txt'
output = dir_ + 'output/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [3]:
def save_report_to_csv(y_true, y_pred, target_names, model_name, filename=None):
    """
    Lưu classification report ra file CSV.
    """
    report_dict = classification_report(y_true, y_pred, target_names=target_names, zero_division=0, output_dict=True)

    df_report = pd.DataFrame(report_dict).transpose()

    df_report['Model'] = model_name
    cols = ['Model'] + [col for col in df_report.columns if col != 'Model']
    df_report = df_report[cols]

    if filename is None:
        clean_model_name = model_name.replace(' ', '_').replace('+', '').lower()
        filename = f"report_{clean_model_name}.csv"

    df_report.to_csv(output + filename, index=True, index_label='Label')
    print(f"✅ Đã lưu báo cáo của mô hình [{model_name}] vào file: {filename}")

    return df_report

In [10]:
df = pd.read_csv(processed_data+'processed.csv')
df = df[df['processed_data'].notna()]
print(df.shape)
df.head(5)

(15778, 11)


,data,stayingpower,texture,smell,price,others,colour,shipping,packing,processed_data,labels
0,Công dụng: tốt\r\nKết cấu: đẹp\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng tốt kết_cấu đẹp độ bền màu lâu,"['stayingpower_positive', 'texture_positive']"
1,Công dụng: son môi\r\nKết cấu: khô\r\nĐộ bền m...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng son môi kết_cấu khô độ bền màu tuyệt...,"['stayingpower_positive', 'texture_positive']"
2,"Son mịn, mùi thơm nhẹ, lâu trôi.\r\nVideo+ hìn...",positive,positive,positive,NaN,NaN,NaN,NaN,NaN,son mịn mùi thơm nhẹ lâu trôi video hình_ảnh m...,"['stayingpower_positive', 'texture_positive', ..."
3,Công dụng: đánh son\r\nKết cấu: Đóng gói cẩn t...,positive,NaN,NaN,positive,NaN,NaN,negative,NaN,công_dụng đánh son kết_cấu đóng_gói cẩn_thận đ...,"['stayingpower_positive', 'price_positive', 's..."
4,Công dụng: tốt\r\nKết cấu: tốt\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,neutral,NaN,NaN,công_dụng tốt kết_cấu tốt độ bền màu tốt đóng_...,"['stayingpower_positive', 'texture_positive', ..."


# Logistic Regression One-vs-Rest kết hợp TF-IDF

In [11]:
aspects = ['stayingpower', 'texture', 'smell', 'price', 'others', 'colour', 'shipping', 'packing']

def extract_labels(row):
    labels = []
    for aspect in aspects:
        if pd.notna(row[aspect]):
            labels.append(f"{aspect}_{row[aspect].strip().lower()}")
    return labels

df['labels'] = df.apply(extract_labels, axis=1)

# Chuyển danh sách nhãn thành ma trận nhị phân
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

X = df['processed_data'].values

print(f"Tổng số mẫu dữ liệu sau khi làm sạch: {len(X)}")
print("Danh sách 24 lớp nhãn:", mlb.classes_)

# Chia tập Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Kích thước tập Train: {X_train.shape[0]}, tập Test: {X_test.shape[0]}")

Tổng số mẫu dữ liệu sau khi làm sạch: 15778
Danh sách 24 lớp nhãn: ['colour_negative' 'colour_neutral' 'colour_positive' 'others_neutral'
 'packing_negative' 'packing_neutral' 'packing_positive' 'price_negative'
 'price_neutral' 'price_positive' 'shipping_negative' 'shipping_neutral'
 'shipping_positive' 'smell_negative' 'smell_neutral' 'smell_positive'
 'stayingpower_negative' 'stayingpower_neutral' 'stayingpower_positive'
 'texture_negative' 'texture_neutral' 'texture_positive']
Kích thước tập Train: 12622, tập Test: 3156


In [12]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 2. Xây dựng và huấn luyện mô hình One-vs-Rest
log_reg = LogisticRegression(solver='liblinear', class_weight='balanced')
ovr_model = OneVsRestClassifier(log_reg)

ovr_model.fit(X_train_tfidf, y_train)

# 3. Dự đoán và Đánh giá
y_pred_ovr = ovr_model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred_ovr, target_names=mlb.classes_, zero_division=0))

df_LR = save_report_to_csv(y_test, y_pred_ovr, mlb.classes_, "logistic_regression_tf_idf")

                       precision    recall  f1-score   support

      colour_negative       0.34      0.77      0.47       134
       colour_neutral       0.22      0.69      0.33       105
      colour_positive       0.89      0.91      0.90      1350
       others_neutral       0.74      0.97      0.84       480
     packing_negative       0.36      0.50      0.42        20
      packing_neutral       0.00      0.00      0.00         2
     packing_positive       0.92      0.95      0.94       600
       price_negative       0.00      0.00      0.00         7
        price_neutral       0.43      0.38      0.40         8
       price_positive       0.96      0.94      0.95       663
    shipping_negative       0.82      0.91      0.86       326
     shipping_neutral       0.23      0.57      0.33        74
    shipping_positive       0.89      0.94      0.91       666
       smell_negative       0.61      0.86      0.71        97
        smell_neutral       0.14      0.53      0.22  

In [9]:
df[df['processed_data'].isna()]

,data,stayingpower,texture,smell,price,others,colour,shipping,packing,processed_data,labels
7446,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
7447,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
7451,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
7452,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
7453,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
...,...,...,...,...,...,...,...,...,...,...,...
7610,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
7611,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
7612,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
7613,NAN,NaN,NaN,NaN,NaN,neutral,NaN,NaN,NaN,NaN,['others_neutral']
